In [37]:
import os

import earthaccess

import virtualizarr

In [2]:
earthaccess.login()

Enter your Earthdata Login username:  aimeeb
Enter your Earthdata password:  ········


In [31]:
# Define the granule query parameters
short_name = "ATL03"
version = "006"
number_of_granules = 1
left_or_right = ["r"]  # ['l', 'r'] # some datasets have both
beams = [1, 2, 3]
groups = [f"gt{beam}{side}" for beam in beams for side in left_or_right]
local_download_folder = f"./{short_name}_local_folder"

In [30]:
groups

['gt1r', 'gt2r', 'gt3r']

# Fetch files over HTTPS

In [ ]:
# Query the granules using CMR (Common Metadata Repository)
granules = earthaccess.search_data(
    short_name=short_name, version=version, count=number_of_granules
)

In [8]:
files = earthaccess.download(granules, local_download_folder)

QUEUEING TASKS | : 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 408.64it/s]
PROCESSING TASKS | : 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:10<00:00, 10.80s/it]
COLLECTING RESULTS | : 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5809.29it/s]


In [26]:
with os.scandir(local_download_folder) as entries:
    files = [
        f"{local_download_folder}/{entry.name}" for entry in entries if entry.is_file()
    ]

# This will only work with one file at a time since different ATL03 granules do not share coordinates
file = files[0]

In [52]:
# vdss - virtual datasets (plural)
vdss = []
drop_variables = [
    "dist_ph_across",
    "dist_ph_along",
    "ph_id_channel",
    "ph_id_count",
    "ph_id_pulse",
    "quality_ph",
    "signal_conf_ph",
    "weight_ph",
    "pce_mframe_cnt",
]
coords = ["delta_time", "lat_ph", "lon_ph"]
for group in groups:
    vds = virtualizarr.open_virtual_dataset(
        filepath=file,
        group=f"/{group}/heights",
        indexes={},
        drop_variables=drop_variables,
    )
    vdss.append(vds.set_coords(coords))

In [58]:
vdss[0].virtualize.to_kerchunk("test.json", format="json")

# Use direct S3 access

Note: This will only work if you are working in the same AWS region as the data (us-west-2).